# olmOCR Table HTML Fine-Tuning on Colab

This notebook is a Colab-ready entrypoint for a first-pass `QLoRA` run on `allenai/olmOCR-2-7B-1025` for table-to-HTML fine-tuning.

It expects the repo files from this workspace to be available in Drive or uploaded into the Colab runtime.

## 1. Install dependencies

In [10]:
!pip install -U transformers trl peft bitsandbytes accelerate datasets pillow scikit-learn beautifulsoup4 qwen-vl-utils

## 2. Mount Google Drive

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Set paths

Update these paths to match where you stored the repo and dataset in Drive.

In [12]:
PROJECT_DIR = '/content/drive/MyDrive/finetune-test'
DATA_JSONL = f'{PROJECT_DIR}/examples/table_html_dataset.example.jsonl'
OUTPUT_DIR = f'{PROJECT_DIR}/outputs/olmocr_table_html_lora'
MAX_SAMPLES = 0
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4

## 4. Inspect runtime

In [13]:
!nvidia-smi

Sun Apr 12 20:49:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
import os
print('PROJECT_DIR exists:', os.path.exists(PROJECT_DIR))
print('DATA_JSONL exists:', os.path.exists(DATA_JSONL))
print('Train script exists:', os.path.exists(f'{PROJECT_DIR}/scripts/train_table_html_lora.py'))

PROJECT_DIR exists: False
DATA_JSONL exists: False
Train script exists: False


## 5. Optional: stage data locally for faster I/O

If Drive is slow, copy the dataset or images into `/content` first.

In [15]:
LOCAL_WORKDIR = '/content/finetune-test'
!rm -rf "$LOCAL_WORKDIR"
!cp -R "$PROJECT_DIR" "$LOCAL_WORKDIR"
PROJECT_DIR = LOCAL_WORKDIR
DATA_JSONL = DATA_JSONL.replace('/content/drive/MyDrive/finetune-test', LOCAL_WORKDIR)
OUTPUT_DIR = '/content/olmocr_table_html_lora'
print(PROJECT_DIR)
print(DATA_JSONL)
print(OUTPUT_DIR)

cp: cannot stat '/content/drive/MyDrive/finetune-test': No such file or directory
/content/finetune-test
/content/finetune-test/examples/table_html_dataset.example.jsonl
/content/olmocr_table_html_lora


## 6. Run a smoke-test training job

In [16]:
!python "$PROJECT_DIR/scripts/train_table_html_lora.py" \
  --data-jsonl "$DATA_JSONL" \
  --output-dir "$OUTPUT_DIR" \
  --max-samples $MAX_SAMPLES \
  --epochs $EPOCHS \
  --batch-size $BATCH_SIZE \
  --grad-accum $GRAD_ACCUM \
  --learning-rate $LEARNING_RATE

python3: can't open file '/content/finetune-test/scripts/train_table_html_lora.py': [Errno 2] No such file or directory


## 7. Save or inspect outputs

In [17]:
!ls -la "$OUTPUT_DIR"

ls: cannot access '/content/olmocr_table_html_lora': No such file or directory


## 8. Optional SSH-style access notes

Colab free managed runtimes can terminate remote shell workflows. If you still want to do it, see `COLAB_SSH_NOTES.md` in the repo for `tmate`, `cloudflared`, and `ngrok` options and caveats.

In [18]:
# Example tmate setup if you already know you want it:
# !apt-get -qq install -y tmate openssh-server
# !service ssh start
# !tmate -F
# !tmate display -p '#{tmate_ssh}'